# HALT — complete pipeline

Everything that produces a number in *Detecting the Generator, Not the
Hallucination*, in one notebook. Each part writes a CSV into the results folder
and prints its own summary.

## What runs where

| Part | Produces | Needs | Time |
|---|---|---|---|
| 0 Setup | locates the data | — | seconds |
| 1 Build benchmark | labels, splits, reliability tiers | CPU | ~10 min |
| 2 Linear baselines | Tables 5, 6, 7, 10 | CPU | ~20 min |
| 3 Label diagnostics | Tables 3, 8, 9 | CPU | ~10 min |
| 4 Shortcut test | Table 2 — **the mechanism** | CPU | ~5 min |
| 5 Bootstrap difference | the interval in §4.4 | CPU | ~10 min |
| 6 Encoder detectors | encoder rows of Tables 2, 10 | **GPU** | ~7 h |
| 7 Prompted-LLM detector | not yet in the paper | **GPU** | ~40 min |
| 8 NLI grounding | not yet in the paper | **GPU** | ~3 h |

Parts 1–6 reproduce the paper. Parts 7 and 8 answer the two experiments it
currently lists as untried.

## Order

Part 1 must run first — everything else reads `benchmark_labels.csv.gz`. After
that, Parts 2–8 are independent and can run in any order or be skipped.

**Set the runtime to GPU** (Runtime → Change runtime type) before Part 6.
Parts 1–5 are CPU-only and will run on any runtime.

## If a run dies

Parts 6, 7 and 8 save after every condition and skip what is already finished.
A Colab disconnect costs only the condition that was running — rerun the same
cell and it continues. Part 8 additionally checkpoints its feature scoring
every 500 reports, since that is the slow half.

---
# Part 0 — Setup

Mounts Drive and locates the results folder. Every later cell reads `OUT` and
`LABELS` from here, so run this once at the start of each session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Direct paths first. The fallback is depth-limited so it cannot walk the whole
# drive over the network mount.
CANDIDATES = [
    '/content/drive/Shareddrives/RESEARCH/2026/PROMPTS/RESULTS',
    '/content/drive/Shareddrives/RESEARCH/Research/2026/PROMPTS/RESULTS',
]

def ok(p):
    return os.path.isdir(os.path.join(p, 'EVALUATION-RESULT'))

RESULTS_ROOT = next((p for p in CANDIDATES if ok(p)), None)

if RESULTS_ROOT is None:
    for base in ['/content/drive/Shareddrives', '/content/drive/MyDrive']:
        if not os.path.isdir(base):
            continue
        depth0 = base.rstrip('/').count('/')
        for root, dirs, _ in os.walk(base):
            if root.count('/') - depth0 >= 4:
                dirs[:] = []
                continue
            if 'RESULTS' in dirs and ok(os.path.join(root, 'RESULTS')):
                RESULTS_ROOT = os.path.join(root, 'RESULTS'); break
        if RESULTS_ROOT:
            break

if RESULTS_ROOT is None:
    raise SystemExit('Not found. Set RESULTS_ROOT by hand.')

RAW = RESULTS_ROOT
OUT = f'{RESULTS_ROOT}/ANALYSIS-OUTPUT-RESULT/iclr_benchmark'
os.makedirs(OUT, exist_ok=True)
os.environ['HALLUBENCH_RAW'] = RAW
os.environ['HALLUBENCH_OUT'] = OUT
LABELS = f'{OUT}/benchmark_labels.csv.gz'

print('RESULTS_ROOT =', RESULTS_ROOT)
print('OUT          =', OUT)
print('labels       =', 'found' if os.path.exists(LABELS) else 'MISSING — run Part 1')

### What can this runtime handle?

Run this before starting a long job. It reports the GPU and says which Parts
will fit, so you find out now rather than five hours into a run.

In [ ]:
import torch, os

print('CPU cores:', os.cpu_count())
if not torch.cuda.is_available():
    print('GPU: none')
    print()
    print('You can run Parts 0-5 (everything the paper needs except the encoders).')
    print('Parts 6-8 need a GPU: Runtime > Change runtime type > GPU.')
else:
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    bf16 = torch.cuda.is_bf16_supported()
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({gb:.0f} GB)  bf16={bf16}')
    print()
    can = {
        'Part 6  DeBERTa 512        (bs 16)': gb >= 12,
        'Part 6  ModernBERT 1,280   (bs 8)' : gb >= 15,
        'Part 6  ModernBERT 2,048   (bs 4)' : gb >= 15,
        'Part 7  Qwen2.5-3B         (bf16)' : gb >= 12,
        'Part 7  Qwen2.5-7B         (bf16)' : gb >= 20 and bf16,
        'Part 8  NLI cross-encoder  (bs 128)': gb >= 12,
    }
    for k, v in can.items():
        print(f'  {"yes" if v else "NO ":4s} {k}')
    if not bf16:
        print()
        print('  No bf16 on this card (pre-Ampere). Part 6 falls back to fp16 with a')
        print('  loss scaler, which is fine. Part 7 needs a smaller model or 8-bit.')
    if gb < 20:
        print()
        print('  For Part 7 on this card, try --model Qwen/Qwen2.5-3B-Instruct.')

---
# Part 0b — Output folder and reruns

**Run this every session, before anything else.**

Results go to `OUT`. Anything retired moves to `OUT/retired/` — nothing is
deleted, and the summary at the end ignores that folder.

Parts 6, 7 and 8 **skip any condition already present in their output file**.
That is what lets a crashed run continue, but it also means a rerun after a fix
will silently do nothing unless the old file is moved first. Set `FORCE` below
to the parts you want to redo.

In [ ]:
import os, shutil, glob

RETIRED = f'{OUT}/retired'
os.makedirs(RETIRED, exist_ok=True)

# Set to True for any part you want to rerun from scratch this session.
FORCE = {
    'encoder':      False,   # Part 6
    'llm_detector': True,    # Part 7  <- set True after fixing the prompt
    'nli':          False,   # Part 8
}

PATTERNS = {
    'encoder':      ['encoder_results*.csv', 'encoder_smoke*.csv'],
    'llm_detector': ['llm_detector*.csv'],
    'nli':          ['nli_results.csv'],          # keeps nli_features.parquet
}

# Always retire these: known-bad or duplicated runs.
ALWAYS = [
    'encoder_results.csv',                   # duplicate of encoder_results_deberta512
    'encoder_results_modernbert_large.csv',  # diverged: random ANY 0.638, a fold at 0.196
]

moved = []
for f in ALWAYS:
    src = f'{OUT}/{f}'
    if os.path.exists(src):
        shutil.move(src, f'{RETIRED}/{f}'); moved.append(f)

for part, on in FORCE.items():
    if not on:
        continue
    for pat in PATTERNS[part]:
        for src in glob.glob(f'{OUT}/{pat}'):
            f = os.path.basename(src)
            shutil.move(src, f'{RETIRED}/{f}'); moved.append(f)

print('moved to retired/:' if moved else 'nothing to retire')
for f in moved:
    print('  ', f)

print('\nActive results in', OUT)
for f in sorted(os.listdir(OUT)):
    if os.path.isfile(f'{OUT}/{f}'):
        print('  ', f)

In [ ]:
!pip -q install pandas scikit-learn scipy openpyxl pyarrow tqdm ipywidgets
import pandas as pd, sklearn
print('pandas', pd.__version__, '| scikit-learn', sklearn.__version__)

---
# Part 1 — Build the benchmark

Rebuilds every label from the raw per-judge verdicts, so nothing downstream
depends on a pre-aggregated file.

**One trap worth knowing.** `full_labeled_dataset_full.csv` aggregates the two
judges disjunctively and reports 98.5% any-hallucination. The released rule is
conjunctive and gives 91.1%. This script derives labels from
`panel_raw_judge_labels_full.csv` only, and uses the other file for report text
alone.

It also creates the four video-disjoint splits and assigns the per-type
reliability tiers from the human-annotated subset.

In [ ]:
%%writefile /content/00_build_benchmark.py
#!/usr/bin/env python3
"""
Step 0 - rebuild canonical labels and construct video-disjoint splits.

Labels come from panel_raw_judge_labels_full.csv under the paper's rule:
two-judge majority, ties broken toward no hallucination. Reproduces the
published corpus statistics (ANY 91.1%, 2.58 hallucinations per report).

Writes: benchmark_labels.csv.gz, label_reliability_tiers.csv,
        benchmark_manifest.json, INTEGRITY_REPORT.md
"""
import json
import numpy as np
import pandas as pd
from sklearn.metrics import cohen_kappa_score

from config import (H, KEY, JOIN, AXIS_OF, AXES, SEED, RAW_DIR, OUT_DIR,
                    JUDGE_FAIL, MULTI_TURN, require, banner)

PAPER = {"any": 91.1, "mean": 2.58}
log_lines = []


def log(m=""):
    print(m)
    log_lines.append(m)


require("panel_raw_judge_labels_full.csv", "full_labeled_dataset_full.csv",
        "Rater-A-scores-807.xlsx", "Rater-B-scores-807.xlsx",
        "Panel-A-scores-807.xlsx")

banner("STEP 0  build benchmark")

# ------------------------------------------------------------ integrity
log("## 1. Source integrity\n")
raw = pd.read_csv(RAW_DIR / "panel_raw_judge_labels_full.csv")
log(f"- raw judge rows: {len(raw):,}")
log(f"- reports: {raw.groupby(KEY).ngroups:,}")
log(f"- judges per report: {raw.groupby(KEY).size().value_counts().to_dict()}")
log(f"- self-judging rows (judge == generator): {(raw.judge == raw.model).sum()}")

sent = (raw[H] == JUDGE_FAIL)
n_cells, n_rows = int(sent.sum().sum()), int(sent.any(axis=1).sum())
log(f"- judge-failure sentinels ({JUDGE_FAIL}): {n_cells} cells / {n_rows} rows "
    f"-> treated as missing, excluded from the vote")
raw[H] = raw[H].mask(sent)

# ---------------------------------------------------------- aggregation
log("\n## 2. Label aggregation (two-judge majority, ties -> negative)\n")
grp = raw.groupby(KEY)[H]
pos, n = grp.sum(min_count=1), grp.count()
lab = (pos >= 2).astype(int)
short = n < 2                                   # a judge failed on this type
lab[short] = (pos[short] >= n[short]).astype(int)
lab = lab.reset_index()

lab["hallucination_count"] = lab[H].sum(axis=1)
lab["any_hallucination"] = (lab.hallucination_count > 0).astype(int)
for ax, cols in AXES.items():
    lab[f"axis_{ax}"] = lab[cols].max(axis=1)

got = (100 * lab.any_hallucination.mean(), lab.hallucination_count.mean())
log(f"- ANY hallucination: {got[0]:.2f}%   (paper {PAPER['any']}%)")
log(f"- mean per report:   {got[1]:.3f}   (paper {PAPER['mean']})")
if abs(got[0] - PAPER["any"]) > 0.5 or abs(got[1] - PAPER["mean"]) > 0.05:
    log("  !! WARNING: does not match the published statistics. Check the "
        "aggregation rule and the input file.")
log("\nPer-model per-type rate (%):\n")
log((100 * lab.groupby("model")[H].mean()).round(1).to_markdown())

# ----------------------------------------------------------- join text
txt = pd.read_csv(RAW_DIR / "full_labeled_dataset_full.csv",
                  usecols=KEY + ["ground_truth", "model_output",
                                 "model_output_full_len"])
bench = lab.merge(txt, on=KEY, how="left", validate="one_to_one")
assert bench.model_output.notna().all(), "text join incomplete"

# --------------------------------------------------- reliability tiers
log("\n## 3. Label reliability against human raters (n=130)\n")
ra = pd.read_excel(RAW_DIR / "Rater-A-scores-807.xlsx")
rb = pd.read_excel(RAW_DIR / "Rater-B-scores-807.xlsx")
pa = pd.read_excel(RAW_DIR / "Panel-A-scores-807.xlsx")

rows = []
tiers = {}
for h in H:
    a, b = ra[f"human_{h}"].to_numpy(), rb[f"human_{h}"].to_numpy()
    pan = pa[f"panel_{h}"].to_numpy()
    agreed = a == b                     # paper's consensus definition
    k_hh = cohen_kappa_score(a, b)
    k_pc = cohen_kappa_score(a[agreed], pan[agreed])
    tier = "high" if k_pc >= 0.65 else ("medium" if k_pc >= 0.40 else "low")
    tiers[h] = tier
    rows.append({"type": h, "axis": AXIS_OF[h], "n_consensus": int(agreed.sum()),
                 "kappa_human_human": round(k_hh, 3),
                 "kappa_panel_consensus": round(k_pc, 3), "tier": tier})
tier_df = pd.DataFrame(rows)
log(tier_df.to_markdown(index=False))
log(f"\n- macro kappa: {tier_df.kappa_panel_consensus.mean():.3f}  (paper 0.556)")
log("- NOTE: these kappas are computed only on rows where both raters agree; "
    "n varies by type and is reported above.")
log(f"- low-reliability types (exclude from headline macro): "
    f"{[h for h in H if tiers[h] == 'low']}")

gold = pa[JOIN].copy()
gold["gold"] = 1
bench = bench.merge(gold, on=JOIN, how="left")
bench["gold"] = bench.gold.fillna(0).astype(int)

# --------------------------------------------------------------- splits
log("\n## 4. Splits (video-disjoint)\n")
rng = np.random.default_rng(SEED)
vmeta = bench[["video", "crime_type"]].drop_duplicates().sort_values("video")
assign = {}
for ct, g in vmeta.groupby("crime_type"):
    v = g.video.to_numpy().copy()
    rng.shuffle(v)
    n_tr, n_va = int(0.70 * len(v)), int(0.15 * len(v))
    for i, name in ((slice(0, n_tr), "train"),
                    (slice(n_tr, n_tr + n_va), "val"),
                    (slice(n_tr + n_va, None), "test")):
        assign.update({x: name for x in v[i]})
bench["split_random"] = bench.video.map(assign)
bench["split_heldout_model"] = np.where(bench.model == "Gemini", "test", "train")
bench["split_heldout_technique"] = np.where(
    bench.technique.isin(MULTI_TURN), "test", "train")

for c in ["split_random", "split_heldout_model", "split_heldout_technique"]:
    leak = int(bench.groupby("video")[c].nunique().gt(1).sum())
    log(f"- {c}: {bench[c].value_counts().to_dict()}   "
        f"videos spanning >1 fold: {leak}")
log(f"\n- gold (human-validated) reports: {int(bench.gold.sum())}")

# ---------------------------------------------------------------- write
cols = (KEY + H + ["hallucination_count", "any_hallucination"]
        + [f"axis_{a}" for a in AXES]
        + ["gold", "split_random", "split_heldout_model",
           "split_heldout_technique", "model_output_full_len",
           "ground_truth", "model_output"])
bench[cols].to_csv(OUT_DIR / "benchmark_labels.csv.gz", index=False,
                   compression="gzip")
tier_df.to_csv(OUT_DIR / "label_reliability_tiers.csv", index=False)
json.dump({"seed": SEED,
           "aggregation": "two-judge majority, ties -> negative",
           "n_reports": int(len(bench)), "n_videos": int(bench.video.nunique()),
           "any_rate": round(float(bench.any_hallucination.mean()), 4),
           "mean_per_report": round(float(bench.hallucination_count.mean()), 4),
           "judge_failure_cells": n_cells, "reliability_tiers": tiers},
          open(OUT_DIR / "benchmark_manifest.json", "w"), indent=2)
(OUT_DIR / "INTEGRITY_REPORT.md").write_text(
    "# Benchmark rebuild integrity report\n\n" + "\n".join(log_lines) + "\n")
print(f"\nwrote -> {OUT_DIR}")


In [ ]:
!python3 /content/00_build_benchmark.py

---
# Part 2 — Linear baselines and leave-one-source-out

Five logistic regressions differing only in features: surface, surface-noid,
tfidf, embed, grounded. Run across the standard split, held-out-strategy, and
each leave-one-source-out fold.

`01` covers the standard and strategy splits; `02` covers leave-one-source-out
with bootstrap confidence intervals and the reference-conditioned features.

In [ ]:
%%writefile /content/01_baselines.py
#!/usr/bin/env python3
"""
Step 1 - baseline detectors.

Feature sets: surface, surface_noid (identity removed), tfidf.
Splits: random, heldout_model, heldout_technique.

Generator identity is dropped automatically on heldout_model and technique
identity on heldout_technique, since those features are constant in training
and unseen at test.

Writes: baseline_results.csv, BASELINE_REPORT.md
"""
import warnings
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

from config import (H, TARGETS, SEED, OUT_DIR, MULTI_TURN, HEDGE_RE,
                    TRUNCATION_CAP, load_benchmark, banner)

warnings.filterwarnings("ignore")
banner("STEP 1  baselines")
df = load_benchmark()

txt = df.model_output
w = txt.str.split().str.len().clip(lower=1)
s = txt.str.count(r"[.!?]").clip(lower=1)
surf = pd.DataFrame({
    "n_words": w, "log_words": np.log1p(w), "n_sents": s,
    "mean_sent_len": w / s,
    "hedge_rate": txt.str.count(HEDGE_RE) / w,
    "detail_rate": txt.str.count(r"\b(\d+|\d{1,2}:\d{2})\b") / w,
    "type_token": txt.str.lower().apply(
        lambda x: len(set(x.split())) / max(len(x.split()), 1)),
    "truncated": (df.model_output_full_len > TRUNCATION_CAP).astype(int),
    "gt_words": df.ground_truth.str.split().str.len(),
})
id_gen = pd.get_dummies(df.model, prefix="gen").astype(float)
id_tec = pd.get_dummies(df.technique, prefix="tec").astype(float)
id_tec["multi_turn"] = df.technique.isin(MULTI_TURN).astype(float)

print("fitting tf-idf ...")
X_tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5,
                          sublinear_tf=True,
                          strip_accents="unicode").fit_transform(txt)

SPLITS = {"random": ("split_random", True, True),
          "heldout_model": ("split_heldout_model", False, True),
          "heldout_technique": ("split_heldout_technique", True, False)}


def build(fs, use_gen, use_tec):
    if fs == "tfidf":
        return X_tfidf
    parts = [surf.to_numpy(float)]
    if fs == "surface":
        if use_gen:
            parts.append(id_gen.to_numpy())
        if use_tec:
            parts.append(id_tec.to_numpy())
    return np.hstack(parts)


from tqdm.auto import tqdm
rows = []
for sname, (scol, ug, ut) in tqdm(SPLITS.items(), total=len(SPLITS),
                                  desc='splits', unit='split'):
    tr, te = (df[scol] == "train").to_numpy(), (df[scol] == "test").to_numpy()
    for fs in ["surface", "surface_noid", "tfidf"]:
        X = build(fs, ug, ut)
        if sparse.issparse(X):
            Xtr, Xte = X[tr], X[te]
        else:
            sc = StandardScaler().fit(X[tr])
            Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        for t in TARGETS:
            y = df[t].to_numpy()
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
                rows.append({"split": sname, "features": fs, "target": t,
                             "auc": np.nan, "f1": np.nan}); continue
            clf = LogisticRegression(max_iter=2000, class_weight="balanced",
                                     random_state=SEED).fit(Xtr, y[tr])
            p = clf.predict_proba(Xte)[:, 1]
            rows.append({"split": sname, "features": fs, "target": t,
                         "auc": roc_auc_score(y[te], p),
                         "f1": f1_score(y[te], (p >= .5).astype(int)),
                         "pos_rate_test": float(y[te].mean())})
        print(f"  {sname} / {fs}")

res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "baseline_results.csv", index=False)

out = ["# Baseline sweep\n",
       "Logistic regression, balanced class weights, seed 42. "
       "All splits video-disjoint. `macro_hq` excludes H1 (low tier).\n"]
for fs in ["surface", "surface_noid", "tfidf"]:
    out.append(f"\n## {fs}\n")
    p = res[res.features == fs].pivot(index="split", columns="target",
                                      values="auc")[TARGETS].round(3)
    p["macro"] = p[H].mean(axis=1).round(3)
    p["macro_hq"] = p[[h for h in H if h != "H1"]].mean(axis=1).round(3)
    out.append(p.to_markdown())
(OUT_DIR / "BASELINE_REPORT.md").write_text("\n".join(out) + "\n")
print("\n".join(out))


In [ ]:
!python3 /content/01_baselines.py

In [ ]:
%%writefile /content/02_logo_grounded.py
#!/usr/bin/env python3
"""
Step 2 - leave-one-generator-out, with grounded and embedding baselines.

Three folds (hold out Claude, GPT, Gemini in turn). Feature sets:
  tfidf              lexical content, ungrounded
  embed              OpenAI embeddings, ungrounded  (skipped if .npy absent)
  grounded           report-vs-reference features only
  grounded+surface   grounded plus the surface block

Writes: logo_results.csv, LOGO_REPORT.md
"""
import warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

from config import (H, TARGETS, AXES, SEED, RAW_DIR, OUT_DIR, JOIN, HEDGE_RE,
                    TRUNCATION_CAP, content_tokens, load_benchmark, banner)

warnings.filterwarnings("ignore")
banner("STEP 2  leave-one-generator-out")
df = load_benchmark()

print("grounded features ...")
rt = [set(content_tokens(x)) for x in df.model_output]
gt = [set(content_tokens(x)) for x in df.ground_truth]
G = pd.DataFrame([{
    "novel_rate": len(r - q) / max(len(r), 1),
    "missing_rate": len(q - r) / max(len(q), 1),
    "jaccard": len(r & q) / max(len(r | q), 1),
    "ref_coverage": len(r & q) / max(len(q), 1),
    "len_ratio": len(r) / max(len(q), 1),
    "log_novel": np.log1p(len(r - q)),
    "log_missing": np.log1p(len(q - r)),
} for r, q in zip(rt, gt)])

tf = TfidfVectorizer(max_features=30000, min_df=3, sublinear_tf=True,
                     strip_accents="unicode", stop_words="english")
tf.fit(pd.concat([df.model_output, df.ground_truth.drop_duplicates()]))
A, B = tf.transform(df.model_output), tf.transform(df.ground_truth)
num = np.asarray(A.multiply(B).sum(axis=1)).ravel()
den = np.sqrt(np.asarray(A.multiply(A).sum(axis=1)).ravel()
              * np.asarray(B.multiply(B).sum(axis=1)).ravel()) + 1e-9
G["tfidf_cos"] = num / den

w = df.model_output.str.split().str.len().clip(lower=1)
S = pd.DataFrame({"n_words": w, "log_words": np.log1p(w),
                  "hedge_rate": df.model_output.str.count(HEDGE_RE) / w,
                  "digit_rate": df.model_output.str.count(r"\d") / w,
                  "truncated": (df.model_output_full_len > TRUNCATION_CAP).astype(int)})

X_tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5,
                          sublinear_tf=True, strip_accents="unicode"
                          ).fit_transform(df.model_output)

FEATS = {"tfidf": lambda: X_tfidf,
         "grounded": lambda: G.to_numpy(float),
         "grounded+surface": lambda: np.hstack([G.to_numpy(float),
                                                S.to_numpy(float)])}

emb_ok = (RAW_DIR / "embeddings_openai.npy").exists() and \
         (RAW_DIR / "embedding_index.csv").exists()
if emb_ok:
    E = np.load(RAW_DIR / "embeddings_openai.npy")
    ix = pd.read_csv(RAW_DIR / "embedding_index.csv")
    ix["emb_row"] = np.arange(len(ix))
    d2 = df.merge(ix[JOIN + ["emb_row"]], on=JOIN, how="left",
                  validate="one_to_one")
    assert d2.emb_row.notna().all(), "embedding index misses reports"
    E = E[d2.emb_row.to_numpy().astype(int)]
    FEATS["embed"] = lambda: E
    print(f"embeddings aligned: {E.shape}")
else:
    print("embeddings not found -> skipping the `embed` baseline")

rows = []
from scipy import sparse
for fname, build in FEATS.items():
    X = build()
    for held in ["Claude", "GPT", "Gemini"]:
        tr, te = (df.model != held).to_numpy(), (df.model == held).to_numpy()
        if sparse.issparse(X):
            Xtr, Xte = X[tr], X[te]
        else:
            sc = StandardScaler().fit(X[tr])
            Xtr, Xte = sc.transform(X[tr]), sc.transform(X[te])
        for t in TARGETS:
            y = df[t].to_numpy()
            if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
                rows.append({"features": fname, "held_out": held,
                             "target": t, "auc": np.nan}); continue
            clf = LogisticRegression(max_iter=3000, class_weight="balanced",
                                     random_state=SEED).fit(Xtr, y[tr])
            rows.append({"features": fname, "held_out": held, "target": t,
                         "auc": roc_auc_score(y[te],
                                              clf.predict_proba(Xte)[:, 1])})
        print(f"  {fname} / held-out {held}")

r = pd.DataFrame(rows)
r.to_csv(OUT_DIR / "logo_results.csv", index=False)

piv = r.pivot_table(index="features", columns="target",
                    values="auc")[TARGETS].round(3)
for ax, cs in AXES.items():
    piv[ax] = piv[cs].mean(axis=1).round(3)
piv["macro"] = piv[H].mean(axis=1).round(3)
out = ["# Leave-one-generator-out\n",
       "AUC on the held-out generator, averaged over three folds.\n",
       piv.to_markdown(), "\n\n## Per-fold detail\n"]
from tqdm.auto import tqdm
for f in tqdm(FEATS, desc='feature sets', unit='set'):
    out.append(f"\n### {f}\n")
    out.append(r[r.features == f].pivot(index="held_out", columns="target",
                                        values="auc")[TARGETS].round(3).to_markdown())
(OUT_DIR / "LOGO_REPORT.md").write_text("\n".join(out) + "\n")
print("\n".join(out))


In [ ]:
!python3 /content/02_logo_grounded.py

---
# Part 3 — Label diagnostics

Three checks on the labels rather than the detectors.

`03` scores each single feature against the panel label and against the human
label on the shared 130-report subset. A large positive gap means the feature
tracks the judge rather than the phenomenon — this produces Table 3, the
length-bias result.

`04` audits the ground-truth sample. `05` runs the within-judge comparison
behind the judge-pair confound: each judge scores exactly two source LLMs, so
holding the judge fixed tests whether the aggregated ordering survives.

In [ ]:
%%writefile /content/03_artifact_test.py
#!/usr/bin/env python3
"""
Step 3 - label-source divergence test.

For each candidate surface feature, compare how well it predicts the PANEL
label against how well it predicts the HUMAN label, on the same 130 reports.
A large positive gap means the feature tracks the judge's decision rule rather
than the phenomenon the humans are scoring.

Also reports panel over-flagging and misses against agreed human labels.

Writes: artifact_test.csv, ARTIFACT_TEST.md
"""
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from config import (H, HNAME, RAW_DIR, OUT_DIR, JOIN, HEDGE_RE,
                    content_tokens, load_benchmark, require, banner)

require("Rater-A-scores-807.xlsx", "Rater-B-scores-807.xlsx",
        "Panel-A-scores-807.xlsx")
banner("STEP 3  artifact test")

df = load_benchmark()
ra = pd.read_excel(RAW_DIR / "Rater-A-scores-807.xlsx")
rb = pd.read_excel(RAW_DIR / "Rater-B-scores-807.xlsx")
pa = pd.read_excel(RAW_DIR / "Panel-A-scores-807.xlsx")

g = pa[JOIN].copy()
for h in H:
    a, b = ra[f"human_{h}"].to_numpy(), rb[f"human_{h}"].to_numpy()
    g[f"panel_{h}"] = pa[f"panel_{h}"].to_numpy()
    g[f"human_{h}"] = a
    g[f"agree_{h}"] = (a == b).astype(int)

m = g.merge(df[JOIN + ["model_output", "ground_truth", "model_output_full_len"]],
            on=JOIN, how="left", validate="one_to_one")
assert m.model_output.notna().all(), "gold reports not found in benchmark table"

rt = [set(content_tokens(x)) for x in m.model_output]
gt = [set(content_tokens(x)) for x in m.ground_truth]
FEATURES = {
    "n_words": m.model_output.str.split().str.len().to_numpy(float),
    "novel_rate": np.array([len(r - q) / max(len(r), 1) for r, q in zip(rt, gt)]),
    "missing_rate": np.array([len(q - r) / max(len(q), 1) for r, q in zip(rt, gt)]),
    "hedge_rate": (m.model_output.str.count(HEDGE_RE)
                   / m.model_output.str.split().str.len().clip(lower=1)).to_numpy(),
}

rows = []
for fname, x in FEATURES.items():
    for h in H:
        yp = m[f"panel_{h}"].to_numpy()
        ok = m[f"agree_{h}"].to_numpy() == 1
        yh = m[f"human_{h}"].to_numpy()[ok]
        ap = roc_auc_score(yp, x) if len(np.unique(yp)) > 1 else np.nan
        ah = roc_auc_score(yh, x[ok]) if len(np.unique(yh)) > 1 else np.nan
        rows.append({"feature": fname, "type": h, "name": HNAME[h],
                     "n_human_agreed": int(ok.sum()),
                     "panel_pos_rate": round(float(yp.mean()), 3),
                     "human_pos_rate": round(float(yh.mean()), 3),
                     "AUC_vs_panel": round(ap, 3), "AUC_vs_human": round(ah, 3),
                     "gap": round(ap - ah, 3)})
res = pd.DataFrame(rows)
res.to_csv(OUT_DIR / "artifact_test.csv", index=False)

out = ["# Label-source divergence\n",
       "AUC of each single feature against the panel label and against the "
       "human label, on the 130 human-validated reports. The human column is "
       "restricted to rows where both raters agree; n is reported per type.\n",
       "A large positive gap means the feature predicts the judge better than "
       "it predicts the humans.\n"]
for fname in FEATURES:
    out.append(f"\n## {fname}\n")
    out.append(res[res.feature == fname][
        ["type", "name", "n_human_agreed", "panel_pos_rate", "human_pos_rate",
         "AUC_vs_panel", "AUC_vs_human", "gap"]].to_markdown(index=False))

out.append("\n\n## Panel behaviour against agreed human labels\n")
rows3 = []
for h in H:
    neg = (m[f"agree_{h}"] == 1) & (m[f"human_{h}"] == 0)
    pos = (m[f"agree_{h}"] == 1) & (m[f"human_{h}"] == 1)
    rows3.append({
        "type": h, "name": HNAME[h],
        "n_human_neg": int(neg.sum()),
        "over_flag_rate": round(float(m.loc[neg, f"panel_{h}"].mean()), 3) if neg.sum() else None,
        "n_human_pos": int(pos.sum()),
        "miss_rate": round(1 - float(m.loc[pos, f"panel_{h}"].mean()), 3) if pos.sum() else None})
out.append(pd.DataFrame(rows3).to_markdown(index=False))
out.append("\n\nCAVEAT: n per cell is small (9 to 110). Treat any single gap "
           "below ~0.10 as noise.")
(OUT_DIR / "ARTIFACT_TEST.md").write_text("\n".join(out) + "\n")
print("\n".join(out))


In [ ]:
!python3 /content/03_artifact_test.py

In [ ]:
%%writefile /content/04_gt_audit.py
#!/usr/bin/env python3
"""
Step 4 - ground-truth audit sample.

UCA annotations are inherited from UCF-Crime's category labels, and at least
one (Abuse002_x264) carries an annotation describing a road-traffic incident.
This script does not decide correctness; it draws a stratified sample for a
human to check, and flags candidates using a crude keyword heuristic whose
false-positive rate is high by design (it is a triage aid, not a measurement).

Writes: gt_audit_sample.csv  (fill in the `verdict` column by hand)
"""
import ast
import json
import numpy as np
import pandas as pd

from config import RAW_DIR, OUT_DIR, SEED, require, banner

banner("STEP 4  ground-truth audit sample")
files = [f for f in ["UCFCrime_Train.json", "UCFCrime_Val.json",
                     "UCFCrime_Test.json"] if (RAW_DIR / f).exists()]
if not files:
    raise SystemExit("No UCFCrime_*.json found in RAW_DIR; skipping audit.")

recs = {}
for f in files:
    d = json.load(open(RAW_DIR / f))
    for vid, v in d.items():
        if isinstance(v, str):
            v = ast.literal_eval(v)
        recs[vid] = {"video": vid,
                     "crime_type": v.get("crime_type", vid.rstrip("0123456789_x264")),
                     "text": " ".join(v.get("sentences", []))}
gtdf = pd.DataFrame(recs.values())
print(f"loaded {len(gtdf)} annotations from {files}")

# Restrict to the eleven crime categories the study actually uses. UCA also
# contains Arrest, Arson and Normal_Videos, which are absent from the 807-video
# corpus and would otherwise be flagged at 100% by the keyword heuristic below
# simply because they have no keyword list.
STUDY = {"Abuse", "Assault", "Burglary", "Explosion", "Fighting",
         "RoadAccidents", "Robbery", "Shooting", "Shoplifting", "Stealing",
         "Vandalism"}
gtdf = gtdf[gtdf.crime_type.isin(STUDY)].copy()
print(f"{len(gtdf)} in the eleven study classes")

KW = {
 "Abuse": ["abuse","beat","hit","slap","kick","punch","push","shov","strangl","drag","throw"],
 "Assault": ["assault","attack","fight","punch","beat","hit","kick","knock","shov"],
 "Burglary": ["burglar","break","broke","pry","climb","window","intrud","steal","stole","door"],
 "Explosion": ["explo","blast","fire","smoke","flame","burst","bomb"],
 "Fighting": ["fight","punch","kick","brawl","beat","hit","wrestl","knock"],
 "RoadAccidents": ["car","vehicle","road","crash","collid","collision","motorcycle","truck","traffic","intersection"],
 "Robbery": ["rob","gun","knife","threat","snatch","demand","cash","money","register"],
 "Shooting": ["shoot","shot","gun","pistol","fire","weapon"],
 "Shoplifting": ["shop","store","shelf","pocket","conceal","steal","stole","merchandise","cashier"],
 "Stealing": ["steal","stole","theft","took","grab","snatch","bag","wallet","pick"],
 "Vandalism": ["vandal","smash","break","broke","destroy","damage","graffiti","kick","shatter"],
}
gtdf["flagged"] = [
    0 if any(k in t.lower() for k in KW.get(c, [])) else 1
    for c, t in zip(gtdf.crime_type, gtdf.text)]
print("\nheuristic flag rate by crime type (HIGH false-positive rate, triage only):")
print((gtdf.groupby("crime_type").flagged.agg(["sum", "count"])
       .assign(pct=lambda d: (100 * d["sum"] / d["count"]).round(0))).to_markdown())

rng = np.random.default_rng(SEED)
flagged = gtdf[gtdf.flagged == 1]
clean = gtdf[gtdf.flagged == 0]
samp = pd.concat([
    flagged.sample(min(25, len(flagged)), random_state=SEED),
    clean.sample(min(25, len(clean)), random_state=SEED)]).sample(frac=1, random_state=SEED)
samp = samp[["video", "crime_type", "flagged", "text"]].copy()
samp["text"] = samp.text.str.slice(0, 600)
samp["verdict"] = ""          # annotator fills: match / mismatch / unclear
samp["notes"] = ""
samp.to_csv(OUT_DIR / "gt_audit_sample.csv", index=False)
print(f"\nwrote {len(samp)} rows -> {OUT_DIR / 'gt_audit_sample.csv'}")
print("Fill the `verdict` column (match / mismatch / unclear). The sample is "
      "balanced 25 flagged / 25 unflagged so you can estimate both error "
      "directions, not just the flagged ones.")


In [ ]:
!python3 /content/04_gt_audit.py

In [ ]:
%%writefile /content/05_judge_confound.py
#!/usr/bin/env python3
"""
Step 5 - judge-pair confound test.

Because no model judges its own output, each generator is scored by a fixed
pair of judges, and those pairs differ sharply in mutual agreement. Under a
conjunctive aggregation rule a disagreeing pair yields fewer positives for
mechanical reasons, so aggregated per-generator rates confound the generator
with its assigned judge pair.

Each judge scores exactly two generators, which permits a within-judge
comparison that holds the judge fixed:

    judge Claude  sees  GPT, Gemini
    judge GPT     sees  Claude, Gemini
    judge Gemini  sees  Claude, GPT

If the generator ordering reported in the aggregated labels is a property of
the generators, every judge should reproduce it on the pair it sees. If it is
a property of the judge pairs, the within-judge orderings will disagree.

Pure analysis of the released per-judge verdicts. No generation, no
re-judging.

Writes: judge_confound.csv, JUDGE_CONFOUND.md
"""
import itertools

import numpy as np
import pandas as pd

from config import H, HNAME, RAW_DIR, OUT_DIR, KEY, JUDGE_FAIL, require, banner

require("panel_raw_judge_labels_full.csv")
banner("STEP 5  judge-pair confound test")

raw = pd.read_csv(RAW_DIR / "panel_raw_judge_labels_full.csv")
raw[H] = raw[H].mask(raw[H] == JUDGE_FAIL)

out = ["# Judge-pair confound test\n",
       "Each generator is scored by a fixed pair of judges. This tests "
       "whether the generator ordering survives when the judge is held "
       "fixed.\n"]

# ---------------------------------------------------- 1. the confound
out.append("\n## 1. Which pair scores which generator\n")
pairs = (raw.groupby("model").judge.unique()
         .apply(lambda a: " + ".join(sorted(a))).rename("judged_by"))
n_rep = raw.groupby("model").size().div(2).astype(int).rename("n_reports")
out.append(pd.concat([pairs, n_rep], axis=1).to_markdown())

# ------------------------------------- 2. aggregated (confounded) view
agg = raw.groupby(KEY)[H]
pos, n = agg.sum(min_count=1), agg.count()
lab = (pos >= 2).astype(int)
short = n < 2
lab[short] = (pos[short] >= n[short]).astype(int)
lab = lab.reset_index()
conf = (100 * lab.groupby("model")[H].mean()).round(1)
out.append("\n\n## 2. Aggregated per-generator rates (confounded)\n")
out.append(conf.to_markdown())

# ------------------------------------------ 3. within-judge comparison
out.append("\n\n## 3. Within-judge rates (%): each judge's own verdicts\n")
wj = (100 * raw.groupby(["judge", "model"])[H].mean()).round(1)
out.append(wj.to_markdown())

# ------------------------------------------------ 4. ordering agreement
out.append("\n\n## 4. Does each judge reproduce the aggregated ordering?\n")
out.append("For every judge and every type, the ordering of the two "
           "generators that judge sees, compared against the ordering the "
           "aggregated labels give for the same two generators.\n")
rows = []
for judge in sorted(raw.judge.unique()):
    seen = sorted(raw.loc[raw.judge == judge, "model"].unique())
    for a, b in itertools.combinations(seen, 2):
        for h in H:
            wa = raw.loc[(raw.judge == judge) & (raw.model == a), h].mean()
            wb = raw.loc[(raw.judge == judge) & (raw.model == b), h].mean()
            ca = lab.loc[lab.model == a, h].mean()
            cb = lab.loc[lab.model == b, h].mean()
            rows.append({
                "judge": judge, "pair": f"{a} vs {b}", "type": h,
                "name": HNAME[h],
                "within_judge": f"{a}" if wa > wb else f"{b}",
                "within_gap_pp": round(100 * abs(wa - wb), 1),
                "aggregated": f"{a}" if ca > cb else f"{b}",
                "agrees": int((wa > wb) == (ca > cb)),
            })
cmp = pd.DataFrame(rows)
cmp.to_csv(OUT_DIR / "judge_confound.csv", index=False)
out.append(cmp[["judge", "pair", "type", "name", "within_judge",
                "within_gap_pp", "aggregated", "agrees"]].to_markdown(index=False))

rate = 100 * cmp.agrees.mean()
out.append(f"\n\n**Orderings preserved: {cmp.agrees.sum()} of {len(cmp)} "
           f"({rate:.0f}%).**\n")
by_h = cmp.groupby("type").agrees.agg(["sum", "count"])
out.append("\nBy type:\n")
out.append(by_h.to_markdown())

# ------------------------------------- 5. does the axis signature hold
out.append("\n\n## 5. Dominant axis, within judge\n")
out.append("Axis rate = mean per-type prevalence within the axis, the "
           "aggregation used for the cross-generation check in the source "
           "study.\n")
AX = {"fabrication": ["H1", "H5", "H6"], "omission": ["H3"],
      "distortion": ["H2", "H4"]}
rows2 = []
for judge in sorted(raw.judge.unique()):
    for gen in sorted(raw.loc[raw.judge == judge, "model"].unique()):
        s = raw[(raw.judge == judge) & (raw.model == gen)]
        r = {"judge": judge, "generator": gen}
        for ax, cs in AX.items():
            r[ax] = round(100 * s[cs].mean().mean(), 1)
        r["dominant"] = max(AX, key=lambda a: r[a])
        rows2.append(r)
ax_df = pd.DataFrame(rows2)
out.append(ax_df.to_markdown(index=False))

out.append("\n\n### Aggregated dominant axis, for comparison\n")
rows3 = []
for gen in sorted(lab.model.unique()):
    s = lab[lab.model == gen]
    r = {"generator": gen}
    for ax, cs in AX.items():
        r[ax] = round(100 * s[cs].mean().mean(), 1)
    r["dominant"] = max(AX, key=lambda a: r[a])
    rows3.append(r)
out.append(pd.DataFrame(rows3).to_markdown(index=False))

(OUT_DIR / "JUDGE_CONFOUND.md").write_text("\n".join(out) + "\n")
print("\n".join(out))


In [ ]:
!python3 /content/05_judge_confound.py

---
# Part 4 — Shortcut test

**The mechanism behind the paper's central claim.** Two measurements:

**(a) Can a classifier tell which LLM wrote a report?** If yes, the shortcut is
available to any detector.

**(b) How far does the source alone get you?** A predictor that reads no report
text — it looks up the writing model and returns that model's training-set
prevalence for the label. Whatever AUC it reaches is attributable to generator
identity alone.

The comparison in (b) is what converts "detectors might be reading style" into
a measured quantity. If the base-rate predictor captures most of the full
detector's above-chance signal on a label, detection of that label is mostly
generator recognition.

In [ ]:
%%writefile /content/08_shortcut_test.py
#!/usr/bin/env python3
"""
Step 8 - does the detector read evidence, or recognise the writer?

Two measurements that test the shortcut directly rather than inferring it from
the transfer result:

  (a) SOURCE CLASSIFICATION. Train a classifier to predict which source LLM
      wrote a report. If this is easy, the shortcut is available.

  (b) BASE-RATE PREDICTOR. A predictor that sees no report text at all: it
      looks up which source LLM wrote the report and returns that source's
      training-set prevalence for the label. Whatever AUC this reaches is
      attributable to source identity alone. Comparing it against the full
      detector, on the above-chance scale, gives the share of the detector's
      signal that source identity supplies.

CPU only, a few minutes. No GPU, no network.

    python3 08_shortcut_test.py

Writes: shortcut_test.csv
"""
import argparse, os
import numpy as np, pandas as pd, warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
warnings.filterwarnings('ignore')

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']
AXIS = {'H1': 'fabrication', 'H5': 'fabrication', 'H6': 'fabrication',
        'H3': 'omission', 'H2': 'distortion', 'H4': 'distortion',
        'any_hallucination': 'binary'}

ap = argparse.ArgumentParser()
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/shortcut_test.csv')
args = ap.parse_args()

df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')

X = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5,
                    sublinear_tf=True, strip_accents='unicode').fit_transform(df.model_output)
tr = (df.split_random == 'train').to_numpy()
te = (df.split_random == 'test').to_numpy()

# ---------------------------------------------------------------- (a)
print('=== (a) SOURCE-MODEL CLASSIFICATION (video-disjoint split) ===')
clf = LogisticRegression(max_iter=3000).fit(X[tr], df.model[tr])
pred = clf.predict(X[te])
acc = accuracy_score(df.model[te], pred)
mf1 = f1_score(df.model[te], pred, average='macro')
print(f'  accuracy  {acc:.3f}   (chance {1/df.model.nunique():.3f})')
print(f'  macro F1  {mf1:.3f}')
for m in sorted(df.model.unique()):
    sel = (df.model[te] == m).to_numpy()
    print(f'    {m:8s} recall {(pred[sel] == m).mean():.3f}')

# ---------------------------------------------------------------- (b)
print('\n=== (b) BASE-RATE PREDICTOR vs FULL DETECTOR ===')
print(f"{'type':6s}{'axis':13s}{'base rate':>10s}{'tfidf':>8s}{'share':>8s}")
rows = []
for t in TARGETS:
    y = df[t].to_numpy()
    if len(np.unique(y[tr])) < 2 or len(np.unique(y[te])) < 2:
        continue
    # base-rate predictor: no text, just the source LLM's training prevalence
    rate = df.loc[tr].groupby('model')[t].mean()
    p_base = df.model[te].map(rate).to_numpy()
    auc_base = roc_auc_score(y[te], p_base)
    # full lexical detector
    d = LogisticRegression(max_iter=2000, class_weight='balanced',
                           random_state=42).fit(X[tr], y[tr])
    auc_full = roc_auc_score(y[te], d.predict_proba(X[te])[:, 1])
    share = (auc_base - 0.5) / (auc_full - 0.5) if auc_full > 0.5 else float('nan')
    rows.append({'target': t, 'axis': AXIS[t], 'auc_base_rate': round(auc_base, 3),
                 'auc_tfidf': round(auc_full, 3), 'share_of_signal': round(share, 3)})
    print(f'{t:6s}{AXIS[t]:13s}{auc_base:10.3f}{auc_full:8.3f}{share:7.0%}')

res = pd.DataFrame(rows)
res.attrs['source_accuracy'] = acc
res.to_csv(args.out, index=False)

fab = res[res.axis == 'fabrication'].share_of_signal
omi = res[res.axis == 'omission'].share_of_signal
print(f'\n  fabrication: {fab.min():.0%}-{fab.max():.0%} of the signal is source identity')
print(f'  omission   : {omi.mean():.0%}')
print('\n  -> that is why fabrication loses most when the source is withheld')
print('\nwrote ->', args.out)


In [ ]:
!python3 /content/08_shortcut_test.py

---
# Part 5 — Bootstrap the degradation difference

The paper reports fabrication degrading 1.4–3.1× more than omission and
distortion. A ratio is a description, not a test.

This resamples the standard-split test set and each leave-one-source-out fold
1,000 times and puts an interval on the difference itself. Run it twice: the
second pass drops H1, the low-reliability type, to confirm the result does not
rest on it.

In [ ]:
%%writefile /content/09_bootstrap_diff.py
#!/usr/bin/env python3
"""
Step 9 - is the degradation difference statistically real?

The paper reports fabrication degrading 1.4 to 3.1 times more than omission and
distortion. A ratio is not a test. This bootstraps the DIFFERENCE

    (fabrication LOSO - fabrication standard) - (other LOSO - other standard)

resampling both the standard-split test set and each leave-one-source-out fold,
and reports a 95% interval. If that interval excludes zero the asymmetry is not
an artifact of sampling.

CPU only, roughly 10 minutes for 1,000 resamples.

    python3 09_bootstrap_diff.py --n-boot 1000

Writes: bootstrap_diff.csv
"""
import argparse, os
import numpy as np, pandas as pd, warnings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
FAB = ['H1', 'H5', 'H6']
OTHER = ['H2', 'H3', 'H4']          # omission + distortion

ap = argparse.ArgumentParser()
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/bootstrap_diff.csv')
ap.add_argument('--n-boot', type=int, default=1000)
ap.add_argument('--drop-h1', action='store_true',
                help='exclude H1, the low-reliability type, from the fabrication axis')
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

fab = [t for t in FAB if not (args.drop_h1 and t == 'H1')]
print('fabrication axis:', fab)

df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
X = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=5,
                    sublinear_tf=True, strip_accents='unicode').fit_transform(df.model_output)


def fit_predict(train_mask, test_mask):
    """Return {type: (y_true, y_score)} on the test set."""
    out = {}
    for t in H:
        y = df[t].to_numpy()
        if len(np.unique(y[train_mask])) < 2 or len(np.unique(y[test_mask])) < 2:
            continue
        c = LogisticRegression(max_iter=2000, class_weight='balanced',
                               random_state=42).fit(X[train_mask], y[train_mask])
        out[t] = (y[test_mask], c.predict_proba(X[test_mask])[:, 1])
    return out


print('fitting standard split ...', flush=True)
P_std = fit_predict((df.split_random == 'train').to_numpy(),
                    (df.split_random == 'test').to_numpy())
P_loso = {}
for held in sorted(df.model.unique()):
    print('fitting LOSO fold', held, flush=True)
    P_loso[held] = fit_predict((df.model != held).to_numpy(),
                               (df.model == held).to_numpy())


def axis_auc(P, idx, types):
    """Mean per-type AUC within an axis, on a bootstrap resample."""
    vals = []
    for t in types:
        if t not in P:
            return None
        y, p = P[t]
        i = idx[t]
        if len(np.unique(y[i])) < 2:
            return None
        vals.append(roc_auc_score(y[i], p[i]))
    return float(np.mean(vals))


rng = np.random.default_rng(args.seed)
diffs, fab_drops, oth_drops = [], [], []
bar = tqdm(range(args.n_boot), desc='bootstrap', unit='resample')
for b in bar:
    idx_s = {t: rng.choice(len(P_std[t][0]), len(P_std[t][0]), replace=True) for t in P_std}
    f_s, o_s = axis_auc(P_std, idx_s, fab), axis_auc(P_std, idx_s, OTHER)
    if f_s is None or o_s is None:
        continue
    f_l, o_l = [], []
    for held in P_loso:
        idx_l = {t: rng.choice(len(P_loso[held][t][0]), len(P_loso[held][t][0]), replace=True)
                 for t in P_loso[held]}
        a = axis_auc(P_loso[held], idx_l, fab)
        c = axis_auc(P_loso[held], idx_l, OTHER)
        if a is None or c is None:
            break
        f_l.append(a); o_l.append(c)
    if len(f_l) < len(P_loso):
        continue
    fd = np.mean(f_l) - f_s
    od = np.mean(o_l) - o_s
    fab_drops.append(fd); oth_drops.append(od); diffs.append(fd - od)
    if len(diffs) % 50 == 0:
        bar.set_postfix(diff=f'{np.mean(diffs):+.3f}')
bar.close()

d = np.array(diffs)
lo, hi = np.percentile(d, [2.5, 97.5])
print()
print('=== fabrication degradation minus omission/distortion degradation (TF-IDF) ===')
print(f'  fabrication drop      {np.mean(fab_drops):+.3f}')
print(f'  omission/distortion   {np.mean(oth_drops):+.3f}')
print(f'  difference            {d.mean():+.3f}')
print(f'  bootstrap 95%         [{lo:+.3f}, {hi:+.3f}]   ({len(d)} resamples)')
print(f'  excludes zero         {hi < 0}')

pd.DataFrame([{'axis_fabrication': '+'.join(fab),
               'fab_drop': round(float(np.mean(fab_drops)), 4),
               'other_drop': round(float(np.mean(oth_drops)), 4),
               'difference': round(float(d.mean()), 4),
               'ci_low': round(float(lo), 4), 'ci_high': round(float(hi), 4),
               'n_resamples': len(d), 'excludes_zero': bool(hi < 0)}]).to_csv(args.out, index=False)
print('\nwrote ->', args.out)


In [ ]:
!python3 /content/09_bootstrap_diff.py --n-boot 1000

In [ ]:
!python3 /content/09_bootstrap_diff.py --n-boot 1000 --drop-h1 \
    --out "$OUT/bootstrap_diff_noH1.csv"

---
# Part 6 — Encoder detectors  **(GPU)**

Fine-tuned encoders, so the transfer result cannot be attributed to
under-powered detectors.

**Context length is a deliberate choice, not a default.** The judge that
produced the labels saw about 1,125 tokens — 3,000 characters of report plus
1,500 of reference. A detector at 512 tokens therefore sees *less than the
judge did*, which is unfair to the detector. Three configurations bracket this:
DeBERTa at its 512 limit, ModernBERT at 1,280 matching the judge, and
ModernBERT at 2,048 exceeding it.

Run the smoke test first. It only checks that the training loop works — do not
read its numbers as results.

In [ ]:
!pip -q install "transformers>=4.48" accelerate sentencepiece
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE — Runtime > Change runtime type > GPU')

In [ ]:
%%writefile /content/06_encoder_detector.py
#!/usr/bin/env python3
"""
Step 6 - fine-tuned encoder detector. RUN THIS ON A GPU (Colab A100 is enough).

Addresses the strongest objection to the paper: that the transfer collapse is
an artifact of linear detectors rather than a property of the task. Fine-tunes
one encoder per split with six per-type heads plus an ANY head, on
[report] [SEP] [reference], and evaluates on the standard, leave-one-source-out
(all three folds), and held-out-strategy splits.

Requires network access to download model weights, so it cannot run in the
offline analysis container.

    pip install "transformers>=4.48" torch scikit-learn accelerate
    python3 06_encoder_detector.py --model answerdotai/ModernBERT-base

CONTEXT LENGTH MATTERS HERE. Reports average ~1,500 tokens. The judge that
produced the labels saw 3,000 characters of report (~750 tokens) plus 1,500 of
reference (~375), so --max_len 1280 gives the detector exactly the judge's
view. At 512 the detector sees less than the judge did and the comparison is
unfair to it. ModernBERT handles 8,192 tokens, so 1,280 costs nothing.

Writes: encoder_results.csv  (same schema as baseline_results.csv, so the
        existing report and figure code consumes it unchanged)

Runtime guide, A100, ModernBERT-base, max_len 1280, bs 8, 2 epochs:
    standard split          ~60 min
    3 LOSO folds            ~150 min
    held-out strategy       ~60 min
Roughly 4.5 h in total. For the validation pass use
    --model roberta-base --max_len 512 --bs 16 --epochs 1
which takes about 20 minutes and only checks that the loop runs.
"""
import argparse, os
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

ap = argparse.ArgumentParser()
ap.add_argument('--model', default='answerdotai/ModernBERT-base')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/encoder_results.csv')
ap.add_argument('--max_len', type=int, default=1280,
                help='1280 matches what the judge saw; raise it to test whether '
                     'the detector benefits from more than the judge had')
ap.add_argument('--epochs', type=int, default=2)
ap.add_argument('--bs', type=int, default=8,
                help='lower than usual because of the long context')
ap.add_argument('--lr', type=float, default=2e-5)
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

torch.manual_seed(args.seed); np.random.seed(args.seed)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
if dev == 'cpu':
    print('WARNING: no GPU visible. This will take many hours.')
else:
    # TF32 is free on Ampere and newer and harmless on older cards.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    _gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({_gb:.0f} GB)  '
          f'bf16={torch.cuda.is_bf16_supported()}')
    _need = {512: 10, 1280: 14, 2048: 16}.get(args.max_len, 16)
    if _gb < _need:
        print(f'  WARNING: --max_len {args.max_len} at --bs {args.bs} wants about '
              f'{_need} GB. Lower --bs or expect an out-of-memory error.')

df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')
tok = AutoTokenizer.from_pretrained(args.model)
_cap = getattr(tok, 'model_max_length', 512)
if _cap and _cap < args.max_len and _cap < 100000:
    print(f'WARNING: {args.model} caps at {_cap} tokens but --max_len is '
          f'{args.max_len}. Reports will be cut below what the judge saw. '
          f'Use a long-context model (ModernBERT, Longformer) or lower '
          f'--max_len and say so in the paper.')


class Reports(Dataset):
    """Report and its reference annotation as a sentence pair."""
    def __init__(self, frame):
        self.a = frame.model_output.tolist()
        self.b = frame.ground_truth.tolist()
        self.y = frame[TARGETS].to_numpy(dtype='float32')

    def __len__(self):
        return len(self.a)

    def __getitem__(self, i):
        # No padding here: the collator pads each batch to its own longest
        # sequence, saving roughly a quarter of the compute at --max_len 2048
        # and costing nothing at shorter windows.
        enc = tok(self.a[i], self.b[i], truncation=True, max_length=args.max_len)
        return ({k: torch.tensor(v) for k, v in enc.items()},
                torch.tensor(self.y[i]))


def collate(batch):
    padded = tok.pad([{k: v.tolist() for k, v in b[0].items()} for b in batch],
                     return_tensors='pt')
    return dict(padded), torch.stack([b[1] for b in batch])


class MultiHead(torch.nn.Module):
    """One shared encoder, seven independent binary heads."""
    def __init__(self, name, n=len(TARGETS)):
        super().__init__()
        # force fp32: some checkpoints (DeBERTa-v3) declare a fp16 dtype in
        # their config, which makes GradScaler refuse to unscale gradients
        self.enc = AutoModel.from_pretrained(name, torch_dtype=torch.float32)
        d = self.enc.config.hidden_size
        self.drop = torch.nn.Dropout(0.1)
        self.heads = torch.nn.Linear(d, n)

    def forward(self, **kw):
        h = self.enc(**kw).last_hidden_state[:, 0]     # [CLS]
        return self.heads(self.drop(h))


def run(train_mask, test_mask, tag):
    tr, te = df[train_mask].reset_index(drop=True), df[test_mask].reset_index(drop=True)
    print(f'\n== {tag}: train {len(tr):,}  test {len(te):,}', flush=True)

    model = MultiHead(args.model).to(dev)
    nw = min(4, os.cpu_count() or 2)
    dl_tr = DataLoader(Reports(tr), batch_size=args.bs, shuffle=True, num_workers=nw,
                       pin_memory=(dev == 'cuda'), collate_fn=collate,
                       persistent_workers=nw > 0)
    dl_te = DataLoader(Reports(te), batch_size=args.bs * 2, num_workers=nw,
                       pin_memory=(dev == 'cuda'), collate_fn=collate,
                       persistent_workers=nw > 0)

    # class weights per head, matching the balanced logistic baselines
    pos = tr[TARGETS].mean().to_numpy()
    w = torch.tensor(((1 - pos) / np.clip(pos, 1e-6, None)).astype('float32')).to(dev)
    lossf = torch.nn.BCEWithLogitsLoss(pos_weight=w)

    opt = torch.optim.AdamW(model.parameters(), lr=args.lr)
    steps = len(dl_tr) * args.epochs
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)

    # bf16 where the GPU supports it (A100 and newer): same dynamic range as
    # fp32, so no loss scaling is needed and GradScaler is skipped entirely.
    use_bf16 = dev == 'cuda' and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
    scaler = torch.amp.GradScaler('cuda', enabled=(dev == 'cuda' and not use_bf16))
    print(f'   precision: {"bf16" if use_bf16 else ("fp16+scaler" if dev=="cuda" else "fp32")}',
          flush=True)

    model.train()
    for ep in range(args.epochs):
        bar = tqdm(dl_tr, desc=f'{tag} ep{ep+1}/{args.epochs}', unit='batch', leave=False)
        run_loss = None
        for x, y in bar:
            x = {k: v.to(dev, non_blocking=True) for k, v in x.items()}
            y = y.to(dev, non_blocking=True)
            opt.zero_grad()
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=(dev == 'cuda')):
                loss = lossf(model(**x), y)
            if scaler.is_enabled():
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
            sch.step()
            l = loss.item()
            run_loss = l if run_loss is None else 0.98 * run_loss + 0.02 * l
            bar.set_postfix(loss=f'{run_loss:.4f}')
        bar.close()
        print(f'   {tag} epoch {ep+1}/{args.epochs} done, loss {run_loss:.4f}', flush=True)

    model.eval(); P = []
    with torch.no_grad():
        for x, _ in tqdm(dl_te, desc=f'{tag} eval', unit='batch', leave=False):
            x = {k: v.to(dev) for k, v in x.items()}
            with torch.amp.autocast('cuda', dtype=amp_dtype, enabled=(dev == 'cuda')):
                P.append(torch.sigmoid(model(**x)).float().cpu().numpy())
    P = np.vstack(P)

    rows = []
    for j, t in enumerate(TARGETS):
        y = te[t].to_numpy()
        if len(np.unique(y)) < 2:
            continue
        rows.append({'features': 'encoder', 'split': tag, 'target': t,
                     'auc': roc_auc_score(y, P[:, j]),
                     'f1': f1_score(y, (P[:, j] >= 0.5).astype(int)),
                     'pos_rate_test': float(y.mean())})
    del model; torch.cuda.empty_cache()
    return rows


# Each condition is saved as soon as it finishes, and any condition already in
# the output file is skipped. A crash or a Colab disconnect costs only the
# condition that was running: rerun the same command to continue.
done = set()
if os.path.exists(args.out):
    done = set(pd.read_csv(args.out).split.unique())
    print(f'resuming: already done -> {sorted(done)}')

conditions = [(df.split_random == 'train', df.split_random == 'test', 'random')]
for held in sorted(df.model.unique()):
    conditions.append((df.model != held, df.model == held, f'heldout_model_{held}'))
conditions.append((df.split_heldout_technique == 'train',
                   df.split_heldout_technique == 'test', 'heldout_technique'))

for train_mask, test_mask, tag in conditions:
    if tag in done:
        print(f'skip {tag} (already saved)')
        continue
    part = pd.DataFrame(run(train_mask, test_mask, tag))
    part.to_csv(args.out, mode='a', header=not os.path.exists(args.out), index=False)
    print(f'   saved {tag} -> {args.out}', flush=True)

res = pd.read_csv(args.out)
res.to_csv(args.out, index=False)
print('\n' + res.pivot_table(index='split', columns='target',
                             values='auc')[TARGETS].round(3).to_markdown())
print('\nwrote ->', args.out)


### Smoke test (~20 min) — loop check only, not a result

In [ ]:
!python3 /content/06_encoder_detector.py --model roberta-base \
    --max_len 512 --bs 16 --epochs 1 --out "$OUT/encoder_smoke.csv"

### DeBERTa-v3-base at 512 (~2.5 h)

In [ ]:
!python3 /content/06_encoder_detector.py --model microsoft/deberta-v3-base \
    --max_len 512 --bs 16 --epochs 2 --out "$OUT/encoder_results_deberta512.csv"

### ModernBERT-base at 1,280 — the judge-matched run (~4.5 h)

In [ ]:
!python3 /content/06_encoder_detector.py --model answerdotai/ModernBERT-base \
    --max_len 1280 --bs 8 --epochs 2 --out "$OUT/encoder_results_modernbert1280.csv"

### ModernBERT-base at 2,048 — more context than the judge had (~5 h)

If this matches the 1,280 run, the ceiling is set by the labels rather than by
what the detector can read.

In [ ]:
!python3 /content/06_encoder_detector.py --model answerdotai/ModernBERT-base \
    --max_len 2048 --bs 4 --epochs 2 --out "$OUT/encoder_results_modernbert2048.csv"

### Sanity check before using any encoder run

A large model that scores *worse in distribution* than a base model has not
trained — check this before reading the leave-one-source-out folds. An AUC
below 0.5 on any fold means divergence, not a finding.

In [ ]:
import pandas as pd, glob, os
for f in sorted(glob.glob(f'{OUT}/encoder_results*.csv')):
    d = pd.read_csv(f)
    r = d[d.split == 'random'].set_index('target').auc
    lo = d[d.split.str.startswith('heldout_model_')].auc.min()
    flag = ''
    if r.get('any_hallucination', 1) < 0.75: flag += '  <-- weak in-distribution'
    if lo < 0.45: flag += '  <-- a fold is anti-predictive'
    print(f"{os.path.basename(f):45s} random ANY {r.get('any_hallucination', float('nan')):.3f}"
          f"  min fold AUC {lo:.3f}{flag}")

---
# Part 7 — Prompted-LLM detector  **(GPU, not yet in the paper)**

The form a practitioner would actually deploy: a model prompted zero-shot with
the report and its reference. Runs locally on the GPU, so **no API key and no
cost**.

**The model must not be Claude, GPT or Gemini.** Those three wrote the corpus;
prompting one as the detector reintroduces the self-preference bias the panel
design exists to exclude, and the number becomes uninterpretable. The script
aborts if the model name looks like one of them.

Zero-shot means no training split — the held-out-source condition is simply the
reports from that source LLM.

Start at 100 per fold and check the `parsed n/n` line. If JSON parsing fails,
fix it there rather than after an hour of generation.

**Rerunning after a fix.** The script skips conditions already present in its
output file, so retire the old results first (the clean-up cell below) or the
rerun will skip everything and change nothing.

The smoke run now prints the positive rate for each type and warns if any is
constant — that is the check that catches a model copying the template rather
than judging. Read it before starting the full run.

**Model size.** 3B fits a 16 GB T4 with room to spare. 7B needs about 15 GB for
weights alone and will offload layers to CPU on a T4, which is slow and can
degrade the output — the setup cell warns you if that is your situation.

**Before rerunning after a fix**, set `FORCE['llm_detector'] = True` in Part 0b
and run that cell. Otherwise this part finds the old output and skips
everything, which is what happens if you only change the script.

In [ ]:
%%writefile /content/10_llm_detector_local.py
#!/usr/bin/env python3
"""
Step 10 - prompted-LLM detector, run LOCALLY on the GPU. No API key, no cost.

Addresses the objection that the paper tests linear and encoder detectors but
not the form a practitioner would actually deploy: a frontier model prompted
zero-shot with the report and its reference.

The model MUST NOT be one of the three that wrote the corpus. Prompting Claude,
GPT or Gemini here would reintroduce the self-preference bias the panel design
exists to exclude, and the number would be uninterpretable. Qwen and Llama are
safe choices and run under `transformers` with no API.

Zero-shot, so there is no training split: the held-out-source condition is
simply the reports from that source LLM.

    pip install "transformers>=4.48" accelerate bitsandbytes
    python3 10_llm_detector_local.py --model Qwen/Qwen2.5-3B-Instruct --n-per-fold 300

Writes: llm_detector_results.csv, llm_detector_raw.csv

RUNTIME. Qwen2.5-3B fits a 16 GB T4 and runs about 1 s per report there; at
--n-per-fold 300 over five conditions that is roughly 25 minutes. The 7B model
needs about 15 GB for weights alone and will offload layers to CPU on a T4,
which is far slower. Start at --n-per-fold 100 to check the prompt and the JSON
parsing before scaling.
"""
import argparse, json, os, re
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score, f1_score
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

RUBRIC = """You are auditing a forensic video report against an expert reference annotation of the same video. For each category decide whether the report contains that error.

H1 scene fabrication: describes a setting or event with no source in the footage
H2 crime misclassification: reports the act as a different category of crime
H3 crime missed: fails to report the criminal act under investigation
H4 severity minimization: understates the gravity of the act
H5 entity fabrication: introduces objects or details not present
H6 phantom actors: introduces people who do not appear

REFERENCE ANNOTATION:
{ref}

REPORT:
{rep}

Use 1 if the error is present and 0 if it is not. Judge each category
independently; most reports contain some errors and not others.

Answer with JSON only, no prose, in exactly this form:
{{"H1": <0 or 1>, "H2": <0 or 1>, "H3": <0 or 1>, "H4": <0 or 1>, "H5": <0 or 1>, "H6": <0 or 1>}}"""

ap = argparse.ArgumentParser()
ap.add_argument('--model', default='Qwen/Qwen2.5-3B-Instruct',
                help='must NOT be Claude, GPT or Gemini')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/llm_detector_results.csv')
ap.add_argument('--raw', default=os.environ.get('HALLUBENCH_OUT', '.') + '/llm_detector_raw.csv')
ap.add_argument('--n-per-fold', type=int, default=300)
ap.add_argument('--batch', type=int, default=8)
ap.add_argument('--max-ref', type=int, default=1500)
ap.add_argument('--max-rep', type=int, default=3000)
ap.add_argument('--seed', type=int, default=42)
args = ap.parse_args()

for banned in ('claude', 'gpt', 'gemini'):
    if banned in args.model.lower():
        raise SystemExit(f'ABORT: {args.model} looks like a source LLM of this corpus. '
                         'Using one as the detector reintroduces self-preference bias.')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
if dev == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    _gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({_gb:.0f} GB)  '
          f'bf16={torch.cuda.is_bf16_supported()}')
    if _gb < 20:
        print('  WARNING: a 7B model in bf16 needs about 15 GB for weights alone. '
              'On a smaller card use a 3B model or load in 8-bit.')
    if not torch.cuda.is_bf16_supported():
        print('  WARNING: this GPU has no bf16. Falling back to fp16.')
print('device:', dev, '| model:', args.model)
tok = AutoTokenizer.from_pretrained(args.model, padding_side='left')
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(
    args.model,
    torch_dtype=(torch.bfloat16 if dev == 'cuda' and torch.cuda.is_bf16_supported()
                 else torch.float16 if dev == 'cuda' else torch.float32),
    device_map='auto')
model.eval()


def parse(txt):
    """Pull the six labels out of the reply. None if unparseable."""
    m = re.search(r'\{[^{}]*\}', txt or '', re.S)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except Exception:
        return None
    if not all(h in d for h in H):
        return None
    try:
        return {h: int(bool(int(d[h]))) for h in H}
    except Exception:
        return None


def score(frame):
    """Score one test set, batched."""
    prompts = [tok.apply_chat_template(
        [{'role': 'user', 'content': RUBRIC.format(
            ref=str(r.ground_truth)[:args.max_ref],
            rep=str(r.model_output)[:args.max_rep])}],
        tokenize=False, add_generation_prompt=True) for _, r in frame.iterrows()]
    out = {}
    bar = tqdm(range(0, len(prompts), args.batch), desc='  generating',
               unit='batch', leave=False)
    for s in bar:
        chunk = prompts[s:s + args.batch]
        enc = tok(chunk, return_tensors='pt', padding=True,
                  truncation=True, max_length=4096).to(dev)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=64, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        for k, g in enumerate(gen):
            reply = tok.decode(g[enc['input_ids'].shape[1]:], skip_special_tokens=True)
            out[frame.index[s + k]] = parse(reply)
        ok = sum(v is not None for v in out.values())
        bar.set_postfix(parsed=f'{ok}/{len(out)}')
    bar.close()
    return out


df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')


def subsample(frame):
    if len(frame) <= args.n_per_fold:
        return frame
    per = max(1, args.n_per_fold // frame.crime_type.nunique())
    return (frame.groupby('crime_type', group_keys=False)
                 .apply(lambda g: g.sample(min(len(g), per), random_state=args.seed)))


conditions = {'random': df[df.split_random == 'test'],
              'heldout_technique': df[df.split_heldout_technique == 'test']}
for held in sorted(df.model.unique()):
    conditions[f'heldout_model_{held}'] = df[df.model == held]

# Each condition is saved as it finishes; rerunning skips what is already there.
done = set()
if os.path.exists(args.out):
    done = set(pd.read_csv(args.out).split.unique())
    print('resuming: already done ->', sorted(done))

rows, raw = [], []
for tag, frame in conditions.items():
    if tag in done:
        print(f'skip {tag} (already saved)')
        continue
    sub = subsample(frame).copy()
    print(f'\n== {tag}: scoring {len(sub):,} reports', flush=True)
    verdicts = score(sub)
    ok = [i for i, v in verdicts.items() if v is not None]
    print(f'   parsed {len(ok)}/{len(sub)}')
    if ok:
        import collections
        rates = {h: np.mean([verdicts[i][h] for i in ok]) for h in H}
        print('   positive rate per type:',
              '  '.join(f'{h} {r:.2f}' for h, r in rates.items()))
        flat = [h for h, r in rates.items() if r in (0.0, 1.0)]
        if flat:
            print(f'   WARNING: {flat} are constant. The model is copying the template '
                  f'rather than judging; the AUCs for those types are meaningless.')
    if not ok:
        print('   NOTHING PARSED - check the prompt or the model before scaling up')
        continue
    sub = sub.loc[ok]
    P = pd.DataFrame([verdicts[i] for i in ok], index=ok)
    P['any_hallucination'] = P[H].max(axis=1)
    raw.append(pd.DataFrame({'split': tag, 'video': sub.video.values,
                             'model': sub.model.values, 'technique': sub.technique.values,
                             **{f'pred_{h}': P[h].values for h in TARGETS},
                             **{f'gold_{h}': sub[h].values for h in TARGETS}}))
    for t in TARGETS:
        y, p = sub[t].to_numpy(), P[t].to_numpy()
        if len(np.unique(y)) < 2:
            continue
        rows.append({'features': 'llm_detector', 'split': tag, 'target': t,
                     'auc': roc_auc_score(y, p), 'f1': f1_score(y, p),
                     'pos_rate_test': float(y.mean()), 'n': len(y)})
    part = pd.DataFrame([r for r in rows if r['split'] == tag])
    part.to_csv(args.out, mode='a', header=not os.path.exists(args.out), index=False)
    raw[-1].to_csv(args.raw, mode='a', header=not os.path.exists(args.raw), index=False)
    print(f'   saved {tag}', flush=True)

res = pd.read_csv(args.out)
print('\n' + res.pivot_table(index='split', columns='target', values='auc')[TARGETS].round(3).to_markdown())
print('\nNOTE: this detector emits hard 0/1 labels, so its AUC is computed on binary')
print('predictions and is NOT directly comparable to the probabilistic baselines.')
print('Report F1 alongside it and say so in the caption.')
print('wrote ->', args.out)


### Smoke test — check parsing before scaling

In [ ]:
!python3 /content/10_llm_detector_local.py --model Qwen/Qwen2.5-3B-Instruct \
    --n-per-fold 100 --out "$OUT/llm_detector_smoke.csv" \
    --raw "$OUT/llm_detector_smoke_raw.csv"

### Full run (~40 min)

In [ ]:
!python3 /content/10_llm_detector_local.py --model Qwen/Qwen2.5-3B-Instruct \
    --n-per-fold 300 --out "$OUT/llm_detector_results.csv" \
    --raw "$OUT/llm_detector_raw.csv"

---
# Part 8 — NLI grounding  **(GPU, not yet in the paper)**

The paper's grounded detector is lexical — token overlap, Jaccard, TF-IDF
cosine. It recovers binary detection under shift but fails on omission (0.577
against TF-IDF's 0.793), and the paper names a semantic version as the most
promising direction not tried.

This scores every report sentence against the reference with a cross-encoder
NLI model, in **both directions**:

- report sentences the reference does not support → fabrication
- reference sentences the report does not carry → **omission**

The second direction is the point. Lexical coverage failed on omission; if
entailment in that direction also fails, the omission gap is not a lexical
problem at all, which is itself worth reporting.

Features cache to parquet, so `--reuse-feats` re-runs the classifiers in
seconds without rescoring.

In [ ]:
%%writefile /content/11_nli_grounding.py
#!/usr/bin/env python3
"""
Step 11 - entailment-based grounded features.

The paper's `grounded` detector is lexical: token overlap, Jaccard, TF-IDF
cosine. It recovers binary detection under shift but fails on omission, and the
paper says a semantic version is the most promising direction not tried. This
is that version.

For each report, every sentence is scored against the reference annotation by a
cross-encoder NLI model, giving per-sentence entailment / neutral /
contradiction probabilities. Those are aggregated into report-level features:

  FABRICATION side - report sentences the reference does not support
    mean/max contradiction of report sentences given the reference
    fraction of report sentences with entailment below a threshold
  OMISSION side - reference content the report does not carry
    the same, with the direction reversed (reference sentences as hypotheses)

The reversed direction is the point: lexical coverage failed on omission, and
entailment in that direction is what should capture "the report drops the
criminal act".

Features then feed the same logistic regression as every other baseline, so the
comparison is like-for-like.

    pip install "transformers>=4.48" accelerate
    python3 11_nli_grounding.py --nli cross-encoder/nli-deberta-v3-base

Writes: nli_features.parquet, nli_results.csv

RUNTIME. 19,361 reports at ~20 sentences each, both directions, is ~800k pairs.
On an A100 in bfloat16 at batch 128 that is roughly 2-3 hours. --max-sent caps
sentences per report; 15 keeps it near 2 hours with little loss.
"""
import argparse, os, re
import numpy as np, pandas as pd, torch, warnings
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

H = ['H1', 'H2', 'H3', 'H4', 'H5', 'H6']
TARGETS = H + ['any_hallucination']

ap = argparse.ArgumentParser()
ap.add_argument('--nli', default='cross-encoder/nli-deberta-v3-base')
ap.add_argument('--labels', default=os.environ.get('HALLUBENCH_OUT', '.') + '/benchmark_labels.csv.gz')
ap.add_argument('--feats', default=os.environ.get('HALLUBENCH_OUT', '.') + '/nli_features.parquet')
ap.add_argument('--out', default=os.environ.get('HALLUBENCH_OUT', '.') + '/nli_results.csv')
ap.add_argument('--max-sent', type=int, default=15, help='sentences per document')
ap.add_argument('--batch', type=int, default=128)
ap.add_argument('--reuse-feats', action='store_true', help='skip scoring, load cached features')
ap.add_argument('--ckpt', type=int, default=500, help='checkpoint every N reports')
args = ap.parse_args()

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
if dev == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    _gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)}  ({_gb:.0f} GB)')
df = pd.read_csv(args.labels)
df['model_output'] = df.model_output.fillna('')
df['ground_truth'] = df.ground_truth.fillna('')

SENT = re.compile(r'(?<=[.!?])\s+')
def sents(t, cap):
    s = [x.strip() for x in SENT.split(str(t)) if len(x.strip()) > 15]
    return s[:cap] if s else ['']

if args.reuse_feats and os.path.exists(args.feats):
    F = pd.read_parquet(args.feats)
    print('loaded cached features', F.shape)
else:
    print('device:', dev, '| NLI model:', args.nli)
    tok = AutoTokenizer.from_pretrained(args.nli)
    nli = AutoModelForSequenceClassification.from_pretrained(
        args.nli, torch_dtype=torch.bfloat16 if dev == 'cuda' else torch.float32).to(dev).eval()
    # label order differs between checkpoints; read it off the config
    id2 = {i: l.lower() for i, l in nli.config.id2label.items()}
    ENT = [i for i, l in id2.items() if 'entail' in l][0]
    CON = [i for i, l in id2.items() if 'contra' in l][0]
    print('  entailment index', ENT, '| contradiction index', CON)

    def score_pairs(premises, hypotheses):
        out = []
        for s in range(0, len(premises), args.batch):
            enc = tok(premises[s:s + args.batch], hypotheses[s:s + args.batch],
                      return_tensors='pt', padding=True, truncation=True,
                      max_length=256).to(dev)
            with torch.no_grad():
                p = torch.softmax(nli(**enc).logits.float(), dim=-1).cpu().numpy()
            out.append(p)
        return np.vstack(out) if out else np.zeros((0, 3))

    # Scoring is the expensive step, so it checkpoints every --ckpt rows.
    # Rerunning picks up where it stopped.
    ckpt = args.feats + '.partial.parquet'
    rows = []
    start = 0
    if os.path.exists(ckpt):
        prev = pd.read_parquet(ckpt)
        rows = prev.to_dict('records'); start = len(rows)
        print(f'resuming feature scoring at row {start:,}')

    bar = tqdm(total=len(df), initial=start, desc='scoring reports', unit='report')
    for n, (_, r) in enumerate(df.iterrows()):
        if n < start:
            continue
        rep_s = sents(r.model_output, args.max_sent)
        ref_s = sents(r.ground_truth, args.max_sent)
        ref_joined = ' '.join(ref_s)[:2000]
        rep_joined = ' '.join(rep_s)[:2000]
        # direction 1: does the reference support each report sentence? (fabrication)
        f = score_pairs([ref_joined] * len(rep_s), rep_s)
        # direction 2: does the report carry each reference sentence? (omission)
        o = score_pairs([rep_joined] * len(ref_s), ref_s)
        rows.append({
            'fab_contra_mean': float(f[:, CON].mean()), 'fab_contra_max': float(f[:, CON].max()),
            'fab_entail_mean': float(f[:, ENT].mean()),
            'fab_unsupported_frac': float((f[:, ENT] < 0.5).mean()),
            'omi_entail_mean': float(o[:, ENT].mean()), 'omi_entail_min': float(o[:, ENT].min()),
            'omi_contra_mean': float(o[:, CON].mean()),
            'omi_dropped_frac': float((o[:, ENT] < 0.5).mean()),
            'n_rep_sent': len(rep_s), 'n_ref_sent': len(ref_s)})
        bar.update(1)
        if (n + 1) % args.ckpt == 0:
            pd.DataFrame(rows).to_parquet(ckpt)
            bar.set_postfix(checkpoint=f'{n+1:,}')
    bar.close()
    F = pd.DataFrame(rows)
    F.to_parquet(args.feats)
    if os.path.exists(ckpt):
        os.remove(ckpt)
    print('wrote features ->', args.feats)

Xn = StandardScaler().fit_transform(F.to_numpy(dtype=float))

def run(train_mask, test_mask, tag):
    out = []
    for t in TARGETS:
        y = df[t].to_numpy()
        if len(np.unique(y[train_mask])) < 2 or len(np.unique(y[test_mask])) < 2:
            continue
        c = LogisticRegression(max_iter=3000, class_weight='balanced',
                               random_state=42).fit(Xn[train_mask], y[train_mask])
        out.append({'features': 'nli_grounded', 'split': tag, 'target': t,
                    'auc': roc_auc_score(y[test_mask], c.predict_proba(Xn[test_mask])[:, 1])})
    return out

res = []
res += run((df.split_random == 'train').to_numpy(), (df.split_random == 'test').to_numpy(), 'random')
for held in sorted(df.model.unique()):
    res += run((df.model != held).to_numpy(), (df.model == held).to_numpy(), f'heldout_model_{held}')
res += run((df.split_heldout_technique == 'train').to_numpy(),
           (df.split_heldout_technique == 'test').to_numpy(), 'heldout_technique')

r = pd.DataFrame(res); r.to_csv(args.out, index=False)
print('\n' + r.pivot_table(index='split', columns='target', values='auc')[TARGETS].round(3).to_markdown())
print('\nThe comparison that matters is omission (H3) under leave-one-source-out:')
print('lexical grounding reaches 0.577 there. If entailment beats that, semantic')
print('grounding is the answer to the omission gap; if not, the gap is not lexical.')
print('wrote ->', args.out)


In [ ]:
!python3 /content/11_nli_grounding.py --nli cross-encoder/nli-deberta-v3-base \
    --max-sent 15 --batch 128

---
# Combined view

Merges every detector family into one table and computes the degradation by
axis. The `ratio` column is the fabrication drop over the larger of the
omission and distortion drops — the conservative choice, quoted in that form
throughout the paper.

In [ ]:
import pandas as pd, numpy as np, glob, os

H = ['H1','H2','H3','H4','H5','H6']; T = H + ['any_hallucination']
AX = {'fabrication': ['H1','H5','H6'], 'omission': ['H3'], 'distortion': ['H2','H4']}

frames = []
for f in sorted(glob.glob(f'{OUT}/*results*.csv')) + [f'{OUT}/baseline_results.csv',
                                                      f'{OUT}/logo_results.csv']:
    if not os.path.exists(f) or 'smoke' in f or '/retired/' in f:
        continue
    d = pd.read_csv(f)
    if 'features' not in d.columns or 'split' not in d.columns:
        continue
    tag = os.path.basename(f).replace('encoder_results_', '').replace('.csv', '')
    if 'encoder' in f:
        d['features'] = tag
    frames.append(d)

allr = pd.concat(frames, ignore_index=True).drop_duplicates(
    subset=['features','split','target'], keep='last')
allr['condition'] = allr.split.str.replace(r'heldout_model_.*', 'heldout_model', regex=True)

piv = allr.pivot_table(index=['features','condition'], columns='target', values='auc')
piv = piv.reindex(columns=T)
for a, cs in AX.items():
    piv[a] = piv[cs].mean(axis=1)
piv['macro'] = piv[H].mean(axis=1)
piv.round(3).to_csv(f'{OUT}/all_detectors_summary.csv')
print(piv.round(3).to_markdown())

rows = []
for feat in piv.index.get_level_values(0).unique():
    try:
        r, l = piv.loc[(feat,'random')], piv.loc[(feat,'heldout_model')]
    except KeyError:
        continue
    other = max(l.omission - r.omission, l.distortion - r.distortion)
    fab = l.fabrication - r.fabrication
    rows.append({'detector': feat, 'fabrication': round(fab,3),
                 'omission': round(l.omission - r.omission,3),
                 'distortion': round(l.distortion - r.distortion,3),
                 'macro': round(l.macro - r.macro,3),
                 'ANY (LOSO)': round(l.any_hallucination,3),
                 'ratio': round(fab/other,2) if other < 0 else None})
deg = pd.DataFrame(rows).set_index('detector')
deg.to_csv(f'{OUT}/degradation_summary.csv')
print('\nDegradation, standard split -> leave-one-source-out\n')
print(deg.to_markdown())

### Files this notebook writes

All under `ANALYSIS-OUTPUT-RESULT/iclr_benchmark/`:

`benchmark_labels.csv.gz` · `label_reliability_tiers.csv` ·
`baseline_results.csv` · `logo_results.csv` · `logo_auc_ci.csv` ·
`logo_strict_results.csv` · `artifact_test.csv` · `gt_audit_sample.csv` ·
`judge_confound.csv` · `shortcut_test.csv` · `bootstrap_diff.csv` ·
`encoder_results_*.csv` · `llm_detector_results.csv` · `nli_results.csv` ·
`all_detectors_summary.csv` · `degradation_summary.csv`